# CSV + API

In this reboot, we are going to use:

- The [Goodreads books](https://www.kaggle.com/jealousleopard/goodreadsbooks) dataset from Kaggle.
- The [Open Library Books API](https://openlibrary.org/dev/docs/api/books)

The goal of this livecode is to load the data from a CSV + loop over rows to enrich each row with information such as:

- List of subjects (Science, Humor, Travel, etc.)
- The cover URL of the book
- Other information you'd find useful in the JSON API

First, download the CSV in the local folder:

In [1]:
!curl -L https://gist.githubusercontent.com/ssaunier/351b17f5a7a009808b60aeacd1f4a036/raw/books.csv > books.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1509k  100 1509k    0     0   901k      0  0:00:01  0:00:01 --:--:--  901k


In [2]:
!ls -lh

total 1.5M
-rw-r--r-- 1 bariscan bariscan  651 Mar 16 14:52 README.md
-rw-r--r-- 1 bariscan bariscan 3.1K Mar 16 14:54 Recap.ipynb
-rw-r--r-- 1 bariscan bariscan 1.5M Mar 16 15:02 books.csv


Then import the usual suspects!

In [3]:
import requests
import pandas as pd
import numpy as np

## Load books from CSV

In [6]:
import pandas as pd

# Veriyi yüklüyoruz 
df = pd.read_csv('books.csv', on_bad_lines='skip') #

# Sütun isimlerini kontrol ederek 'isbn' sütununun varlığından emin olalım
print(df.columns)

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '# num_pages', 'ratings_count', 'text_reviews_count'],
      dtype='object')


In [7]:
import requests

def fetch_book_data(isbn):
    """
    Verilen ISBN numarası ile Open Library API'sinden kitap bilgilerini çeker.
    """
    # API URL'sini oluşturuyoruz (format=json ve jscmd=data detaylı veri sağlar)
    url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{isbn}&format=json&jscmd=data"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        # API yanıtı içindeki asıl veri bloğuna erişiyoruz
        key = f"ISBN:{isbn}"
        if key in data:
            book_info = data[key]
            
            # İhtiyacımız olan alanları ayıklıyoruz
            subjects = [s['name'] for s in book_info.get('subjects', [])]
            cover_url = book_info.get('cover', {}).get('large', "Kapak Yok")
            
            return {
                'Subjects': subjects,
                'Cover_URL': cover_url,
                'OpenLibrary_URL': book_info.get('url', "")
            }
    except Exception as e:
        print(f"Hata: {isbn} için veri çekilemedi. {e}")
        
    return {'Subjects': [], 'Cover_URL': "Hata", 'OpenLibrary_URL': ""}

# Test: İlk kitaba ait ISBN ile fonksiyonu deneyelim
test_isbn = df.iloc[0]['isbn']
print(f"Test ISBN: {test_isbn}")
print(fetch_book_data(test_isbn))

Test ISBN: 0439785960
{'Subjects': ['orphans', 'foster homes', 'romans', 'magie', 'adolescence', 'Quill Award winner', 'Scottish Children’s Book Award winner', 'British Book of the Year Award winner', 'Fiction', 'Juvenile fiction', 'Magic', 'Schools', 'Witches', 'Wizards', 'New York Times bestseller', 'Fantasy fiction', 'nyt:series_books=2006-07-15', 'nyt:series_books=2006-09-16', 'Romans, nouvelles, etc. pour la jeunesse', 'Sorciers', 'Roman fantastique', 'Merveilleux', 'Hogwarts School of Witchcraft and Wizardry (Imaginary place)', 'Harry Potter (Fictitious character)', 'Hogwarts School of Witchcraft and Wizardry (Imaginary organization)', 'Magos', 'Magia', 'Ficción juvenil', 'Escuelas', 'Novela fantástica', 'England', 'School stories', 'Family', 'Harry Potter (Fictional character)', 'Orphans & Foster Homes', 'Social Themes', 'Fantasy', 'Fantasy & Magic', 'Friendship', 'Reading Level-Grade 11', 'Reading Level-Grade 10', 'Reading Level-Grade 12', 'England, fiction', 'Magic, fiction', 

Let's add a new column

## API - Open Library

In [8]:
# 1. API limitlerine takılmamak ve hızlı sonuç görmek için ilk 15 satırlık bir örnek oluşturalım
# Tüm veriseti (11.000 satır) için işlem saatler sürebilir.
df_enriched = df.head(15).copy()

print("API'den veriler çekiliyor, lütfen bekleyin...")

# 2. 'isbn' sütununu kullanarak fonksiyonu her satıra uygulayalım
# Bu işlem her satır için bir sözlük (dict) döndürecektir.
df_enriched['api_data'] = df_enriched['isbn'].apply(fetch_book_data)

# 3. Dönen sözlükteki verileri ayrı sütunlara dağıtalım
df_enriched['Subjects'] = df_enriched['api_data'].apply(lambda x: x.get('Subjects'))
df_enriched['Cover_URL'] = df_enriched['api_data'].apply(lambda x: x.get('Cover_URL'))
df_enriched['OpenLibrary_URL'] = df_enriched['api_data'].apply(lambda x: x.get('OpenLibrary_URL'))

# 4. Geçici olarak oluşturduğumuz 'api_data' sütununu silelim
df_enriched.drop(columns=['api_data'], inplace=True)

print("İşlem tamamlandı! Zenginleştirilmiş veri seti hazır.")

# Sonucu görüntüleyelim
df_enriched[['title', 'isbn', 'Subjects', 'Cover_URL', 'OpenLibrary_URL']]

API'den veriler çekiliyor, lütfen bekleyin...
İşlem tamamlandı! Zenginleştirilmiş veri seti hazır.


,title,isbn,Subjects,Cover_URL,OpenLibrary_URL
0,Harry Potter and the Half-Blood Prince (Harry ...,0439785960,"[orphans, foster homes, romans, magie, adolesc...",https://covers.openlibrary.org/b/id/15156081-L...,https://openlibrary.org/books/OL24280830M/Harr...
1,Harry Potter and the Order of the Phoenix (Har...,0439358078,"[Children's Books/Ages 9-12 Fiction, Witches a...",https://covers.openlibrary.org/b/id/12025650-L...,https://openlibrary.org/books/OL34152967M/Harr...
2,Harry Potter and the Sorcerer's Stone (Harry P...,0439554934,"[series:Harry_Potter, Ghosts, Monsters, Vampir...",https://covers.openlibrary.org/b/id/7572543-L.jpg,https://openlibrary.org/books/OL26018592M/Harr...
3,Harry Potter and the Chamber of Secrets (Harry...,0439554896,"[series:Harry_Potter, Fantasy fiction, school ...",https://covers.openlibrary.org/b/id/10301720-L...,https://openlibrary.org/books/OL28361807M/Harr...
4,Harry Potter and the Prisoner of Azkaban (Harr...,043965548X,"[Fantasy fiction, orphans, foster homes, fanta...",https://covers.openlibrary.org/b/id/8778528-L.jpg,https://openlibrary.org/books/OL27305590M/Harr...
5,Harry Potter Boxed Set Books 1-5 (Harry Potte...,0439682584,"[Potter, harry (fictitious character), fiction...",https://covers.openlibrary.org/b/id/278981-L.jpg,https://openlibrary.org/books/OL7514736M/Harry...
6,"Unauthorized Harry Potter Book Seven News: ""Ha...",0976540606,"[Characters, Harry Potter, Children's stories,...",https://covers.openlibrary.org/b/id/742235-L.jpg,https://openlibrary.org/books/OL8588088M/Unaut...
7,Harry Potter Collection (Harry Potter #1-6),0439827604,"[England, fiction, Fantasy fiction, Magic, fic...",https://covers.openlibrary.org/b/id/279436-L.jpg,https://openlibrary.org/books/OL7515755M/Harry...
8,The Ultimate Hitchhiker's Guide: Five Complete...,0517226952,"[comic science fiction, Vogons, Humorous ficti...",https://covers.openlibrary.org/b/id/12617870-L...,https://openlibrary.org/books/OL3420524M/The_U...
9,The Ultimate Hitchhiker's Guide to the Galaxy,0345453743,"[comic science fiction, Vogons, Humorous ficti...",https://covers.openlibrary.org/b/id/14656530-L...,https://openlibrary.org/books/OL17044900M/The_...


## Calling the API with multiple ISBNs at a time

In [9]:
# Birden fazla ISBN'i tek seferde sorgulayan optimize kod
def fetch_multiple_books(isbn_list):
    # ISBN listesini 'ISBN:1234,ISBN:5678' formatına getiriyoruz
    bibkeys = ",".join([f"ISBN:{isbn}" for isbn in isbn_list])
    url = f"https://openlibrary.org/api/books?bibkeys={bibkeys}&format=json&jscmd=data"
    
    response = requests.get(url)
    return response.json()

# Örnek uygulama: İlk 5 kitabı tek bir API çağrısı ile alalım
sample_isbns = df['isbn'].head(5).tolist()
multi_data = fetch_multiple_books(sample_isbns)

print(f"Tek seferde çekilen kitap sayısı: {len(multi_data)}")

Tek seferde çekilen kitap sayısı: 5


In [10]:
import requests
import pandas as pd
import time

# 1. Çoklu ISBN sorgulayan optimize fonksiyon
def fetch_multiple_books(isbn_list):
    """
    Birden fazla ISBN numarasını tek bir API isteğiyle sorgular.
    """
    # ISBN listesini 'ISBN:0439785960,ISBN:0439358078...' formatına çeviriyoruz
    bibkeys = ",".join([f"ISBN:{isbn}" for isbn in isbn_list])
    url = f"https://openlibrary.org/api/books?bibkeys={bibkeys}&format=json&jscmd=data"
    
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
    except Exception as e:
        print(f"Hata oluştu: {e}")
    return {}

# 2. Örnek uygulama: İlk 100 kitabı 20'şerli gruplar halinde çekelim
df_sample = df.head(100).copy()
batch_size = 20
all_enriched_data = {}

print(f"{len(df_sample)} kitap için toplu işlem başlatılıyor...")

for i in range(0, len(df_sample), batch_size):
    # Mevcut batch için ISBN listesini al
    batch_isbns = df_sample['isbn'].iloc[i : i + batch_size].tolist()
    print(f"Batch {i//batch_size + 1} çekiliyor...")
    
    # API'den toplu veriyi al ve ana sözlüğe ekle
    batch_data = fetch_multiple_books(batch_isbns)
    all_enriched_data.update(batch_data)
    
    # API limitlerine takılmamak için kısa bir bekleme
    time.sleep(1)

# 3. Çekilen verileri DataFrame'e geri haritalama (Mapping)
def extract_from_batch(isbn, data_store):
    key = f"ISBN:{isbn}"
    if key in data_store:
        book_info = data_store[key]
        return {
            'Subjects': [s['name'] for s in book_info.get('subjects', [])],
            'Cover_URL': book_info.get('cover', {}).get('large', "N/A"),
            'OL_Link': book_info.get('url', "")
        }
    return {'Subjects': [], 'Cover_URL': "N/A", 'OL_Link': ""}

# Yeni verileri listeye al ve DataFrame ile birleştir
enriched_results = [extract_from_batch(isbn, all_enriched_data) for isbn in df_sample['isbn']]
enriched_df = pd.DataFrame(enriched_results)

# Orijinal tablo ile yan yana getir
df_final = pd.concat([df_sample.reset_index(drop=True), enriched_df], axis=1)

print("Toplu zenginleştirme tamamlandı!")
df_final[['title', 'isbn', 'Subjects', 'Cover_URL']].head()

100 kitap için toplu işlem başlatılıyor...
Batch 1 çekiliyor...
Batch 2 çekiliyor...
Batch 3 çekiliyor...
Batch 4 çekiliyor...
Batch 5 çekiliyor...
Toplu zenginleştirme tamamlandı!


,title,isbn,Subjects,Cover_URL
0,Harry Potter and the Half-Blood Prince (Harry ...,0439785960,"[orphans, foster homes, romans, magie, adolesc...",https://covers.openlibrary.org/b/id/15156081-L...
1,Harry Potter and the Order of the Phoenix (Har...,0439358078,"[Children's Books/Ages 9-12 Fiction, Witches a...",https://covers.openlibrary.org/b/id/12025650-L...
2,Harry Potter and the Sorcerer's Stone (Harry P...,0439554934,"[series:Harry_Potter, Ghosts, Monsters, Vampir...",https://covers.openlibrary.org/b/id/7572543-L.jpg
3,Harry Potter and the Chamber of Secrets (Harry...,0439554896,"[series:Harry_Potter, Fantasy fiction, school ...",https://covers.openlibrary.org/b/id/10301720-L...
4,Harry Potter and the Prisoner of Azkaban (Harr...,043965548X,"[Fantasy fiction, orphans, foster homes, fanta...",https://covers.openlibrary.org/b/id/8778528-L.jpg


In [12]:
df_final.to_csv("enriched_books_final.csv", index=False)
print("Dosya başarıyla oluşturuldu!")

Dosya başarıyla oluşturuldu!
